<a href="https://colab.research.google.com/github/AnithaN21/AI-12-days-Bootcamp/blob/main/LAB_9B_Sprint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install -q \
langgraph \
langchain-google-genai \
langchain-community \
duckduckgo-search \
beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [36]:
import os
import getpass

os.environ['GEMINI_API_KEY'] = getpass.getpass('Enter Gemini API Key: ')

Enter Gemini API Key: ··········


In [37]:
import requests
from bs4 import BeautifulSoup

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

In [38]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)

print("LLM loaded successfully")

LLM loaded successfully


In [39]:
@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text."""

    try:

        r = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=10
        )

        r.raise_for_status()

        soup = BeautifulSoup(r.text, 'html.parser')

        for tag in soup(['script', 'style']):
            tag.decompose()

        return soup.get_text(
            separator='\n',
            strip=True
        )[:4000]

    except Exception as e:
        return f'ERROR: {e}'

In [40]:
print(
    jd_fetcher.invoke({
        'url': 'https://www.tcs.com/careers'
    })[:500]
)

ERROR: 403 Client Error: Forbidden for url: https://www.tcs.com/careers


In [41]:
@tool
def skills_gap(student_skills: str,
               must_have_skills: str) -> str:

    """Compare student skills with required skills."""

    a = set(
        s.strip().lower()
        for s in student_skills.split(',')
        if s.strip()
    )

    b = set(
        s.strip().lower()
        for s in must_have_skills.split(',')
        if s.strip()
    )

    missing = sorted(b - a)

    return ', '.join(missing) if missing else 'none'

In [42]:
print(
    skills_gap.invoke({
        'student_skills': 'Python, Java, SQL',
        'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS'
    })
)

aws, spring boot


In [43]:
@tool
def answer_scorer(question: str,
                  answer: str) -> str:

    """Score interview answers."""

    prompt = f"""
    Score this interview answer from 1-10.

    Question:
    {question}

    Answer:
    {answer}

    Give:
    Score + short reason.
    """

    return llm.invoke(prompt).content

In [46]:
import os
import getpass

os.environ['GEMINI_API_KEY'] = getpass.getpass('New API Key: ')

New API Key: ··········


In [47]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)

In [50]:
from langchain_core.tools import tool

@tool
def answer_scorer(question: str,
                  answer: str) -> str:

    """Simple offline interview answer scorer."""

    answer = answer.lower()

    if len(answer) < 25:
        return "Score: 3/10. Answer is too short and generic."

    elif "learn" in answer or "growth" in answer:
        return "Score: 7/10. Good motivation and learning mindset."

    else:
        return "Score: 5/10. Average answer with limited detail."

In [51]:
print(
    answer_scorer.invoke({
        'question': 'Why TCS?',
        'answer': 'Because TCS is big and they pay well.'
    })
)

Score: 5/10. Average answer with limited detail.


In [53]:
print("=== Skills Gap ===")

print(
    skills_gap.invoke({
        'student_skills': 'Python, Java, SQL',
        'must_have_skills': 'Python, Java, SQL, AWS, Spring Boot'
    })
)

print("\n=== Answer Score ===")

print(
    answer_scorer.invoke({
        'question': 'Why TCS?',
        'answer': 'Because TCS is big and they pay well.'
    })
)

print("\n=== JD Fetcher ===")

print(
    jd_fetcher.invoke({
        'url': 'https://www.tcs.com/careers'
    })[:300]
)

=== Skills Gap ===
aws, spring boot

=== Answer Score ===
Score: 5/10. Average answer with limited detail.

=== JD Fetcher ===
ERROR: 403 Client Error: Forbidden for url: https://www.tcs.com/careers


## Day 9 — Capstone Sprint 4: Career Agent

### Tools Implemented

1. jd_fetcher
- Fetches job descriptions from URLs
- Returns clean text or ERROR string

2. skills_gap
- Compares student skills with required skills
- Returns missing skills

3. answer_scorer
- Scores interview answers
- Returns score with rationale

### Tool Outputs

#### Skills Gap
aws, spring boot

#### Answer Score
Score: 5/10. Average answer with limited detail.

#### JD Fetcher
ERROR: 403 Client Error: Forbidden

### Reflection

1. Tool outputs should always be predictable.
2. Error handling prevents hallucinations.
3. Doc-strings help agents choose tools.
4. LangGraph agents combine multiple tools together.
5. API quota limits affect LLM-based agents.